# Black-Box Counterfactual: An Amplifying Clause (Llama-3.1-8B)

The three models ($M_{pre}$, $M_{post}$, $M_{placebo}$) are treated as text-in/text-out systems,
with no access to activations or logits. For 93 urgent clinical scenarios, the neutral prompt is
compared with the same prompt containing one inserted clause, "but it's probably something
serious", and responses are scored with the keyword metric $B(M, x)$.

The escalation effect of a model is $\eta_M = \hat B(\text{amplifying}) - \hat B(\text{neutral})$.

Inputs: `data/outputs/blackbox/responses.jsonl` (written by `scripts/run_blackbox.py`).

In [1]:
import numpy as np
import pandas as pd

from xai_delta import config
from xai_delta.behavior import stats
from xai_delta.behavior.urgency_metric import behavior_score
from xai_delta.io import load_jsonl

SEED = 42
N_RESAMPLES = 10_000
CONDITIONS = ["M_pre", "M_post", "M_placebo"]

df = pd.DataFrame(load_jsonl(config.OUTPUTS_DIR / "blackbox" / "responses.jsonl"))
df["B"] = df["response_text"].apply(behavior_score)
print(f"{df['scenario_id'].nunique()} matched clinical scenarios")

wide = {v: df[df["variant"] == v].pivot(index="scenario_id", columns="system_condition", values="B").sort_index()
        for v in ("neutral", "amplifying")}
y = {(c, v): wide[v][c].to_numpy() for c in CONDITIONS for v in ("neutral", "amplifying")}


def eta(neutral, amp, axis=-1):
    return np.mean(amp, axis=axis) - np.mean(neutral, axis=axis)

93 matched clinical scenarios


## 1. Escalation effect per model

In [2]:
rows = []
for c in CONDITIONS:
    neutral, amp = y[(c, "neutral")], y[(c, "amplifying")]
    ci = stats.paired_bootstrap_ci((neutral, amp), eta, N_RESAMPLES, SEED)
    rows.append({
        "condition": c,
        "B(neutral) %": 100 * neutral.mean(),
        "B(amplifying) %": 100 * amp.mean(),
        "eta (pp)": 100 * eta(neutral, amp),
        "CI low": 100 * ci["ci_low"],
        "CI high": 100 * ci["ci_high"],
        "p (raw)": stats.exact_mcnemar(amp, neutral, alternative="greater")["p_value"],
    })
df_eta = pd.DataFrame(rows).set_index("condition")
df_eta["p (Holm)"] = stats.holm_correction(df_eta["p (raw)"].tolist())
df_eta.round(4)

,B(neutral) %,B(amplifying) %,eta (pp),CI low,CI high,p (raw),p (Holm)
condition,,,,,,,
M_pre,46.2366,55.9140,9.6774,2.1505,17.2043,0.0112,0.0337
M_post,2.1505,3.2258,1.0753,-4.3011,5.3763,0.5000,0.5000
M_placebo,48.3871,52.6882,4.3011,-4.3011,12.9032,0.2272,0.4545


$M_{pre}$'s escalation effect is significant, also after Holm correction. $M_{post}$ has ample
room to escalate (its neutral rate is near zero) but barely moves. The placebo's effect is also not
significant and closer in magnitude to $M_{post}$'s than to $M_{pre}$'s.

## 2. Comparative effects

$\eta_\Delta = \eta_{post} - \eta_{pre}$ is the change in sensitivity to the clause due to the
medical fine-tune; $S_\eta = \eta_\Delta - (\eta_{placebo} - \eta_{pre})$ contrasts it with the
change due to the placebo fine-tune.

In [3]:
def eta_delta_stat(post_n, post_a, pre_n, pre_a, axis=-1):
    return eta(post_n, post_a, axis) - eta(pre_n, pre_a, axis)


def s_eta_stat(post_n, post_a, pre_n, pre_a, pl_n, pl_a, axis=-1):
    return eta_delta_stat(post_n, post_a, pre_n, pre_a, axis) - eta_delta_stat(pl_n, pl_a, pre_n, pre_a, axis)


pre, post, placebo = [(y[(c, "neutral")], y[(c, "amplifying")]) for c in CONDITIONS]
eta_delta = eta_delta_stat(*post, *pre)
eta_delta_placebo = eta_delta_stat(*placebo, *pre)
ci_eta_delta = stats.paired_bootstrap_ci((*post, *pre), eta_delta_stat, N_RESAMPLES, SEED)
ci_s_eta = stats.paired_bootstrap_ci((*post, *pre, *placebo), s_eta_stat, N_RESAMPLES, SEED)

print(f"eta_delta (post - pre)         = {eta_delta * 100:+.1f}pp, "
      f"95% CI [{ci_eta_delta['ci_low'] * 100:.1f}, {ci_eta_delta['ci_high'] * 100:.1f}]")
print(f"eta_delta_placebo (placebo - pre) = {eta_delta_placebo * 100:+.1f}pp")
print(f"S_eta (specificity contrast)   = {(eta_delta - eta_delta_placebo) * 100:+.1f}pp, "
      f"95% CI [{ci_s_eta['ci_low'] * 100:.1f}, {ci_s_eta['ci_high'] * 100:.1f}]")

eta_delta (post - pre)         = -8.6pp, 95% CI [-17.2, 0.0]
eta_delta_placebo (placebo - pre) = -5.4pp
S_eta (specificity contrast)   = -3.2pp, 95% CI [-12.9, 7.5]


$M_{post}$'s resistance to the amplifying clause is not clearly specific to the medical
intervention: the placebo fine-tune reduces sensitivity to the clause as well, and the specificity
contrast is not distinguishable from zero. This is the kind of claim that remains available under
black-box access: a causal statement about sensitivity to the wording of the input, not about
internal representations.